In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import confusion_matrix
from xgboost import XGBClassifier

# ============================================
# DEFINING PERFORMANCE METRICS FUNCTION
# ============================================
def performance_measures(pred_class, y_true, classes=[1, 2, 3, 4, 5]):
    tp=np.zeros(len(classes), dtype=int)
    fp=np.zeros(len(classes), dtype=int)
    tn=np.zeros(len(classes), dtype=int)
    fn=np.zeros(len(classes), dtype=int)

    accuracy=np.full(len(classes), np.nan)
    sensitivity=np.full(len(classes), np.nan)
    specificity=np.full(len(classes), np.nan)
    precision=np.full(len(classes), np.nan)
    f1_score=np.full(len(classes), np.nan)

    for i, cls in enumerate(classes):
        tp_class=((pred_class==cls) & (y_true==cls)).astype(int)
        fp_class=((pred_class==cls) & (y_true!=cls)).astype(int)
        tn_class=((pred_class!=cls) & (y_true!=cls)).astype(int)
        fn_class=((pred_class!=cls) & (y_true==cls)).astype(int)

        tp[i]=tp_class.sum()
        fp[i]=fp_class.sum()
        tn[i]=tn_class.sum()
        fn[i]=fn_class.sum()

        denom=tp[i]+fp[i]+tn[i]+fn[i]
        accuracy[i]=(tp[i]+tn[i])/denom if denom != 0 else np.nan
        sensitivity[i]=tp[i]/(tp[i]+fn[i]) if (tp[i]+fn[i]) != 0 else np.nan
        specificity[i]=tn[i]/(tn[i]+fp[i]) if (tn[i]+fp[i]) != 0 else np.nan
        precision[i]=tp[i]/(tp[i]+fp[i]) if (tp[i]+fp[i]) != 0 else np.nan

        if (np.isnan(precision[i]) or np.isnan(sensitivity[i])
            or (precision[i]+sensitivity[i])==0):
            f1_score[i]=np.nan
        else:
            f1_score[i]=2*tp[i]/(2*tp[i]+fn[i]+fp[i])

        print(f"Class Performance Measures for Class: {cls}")
        print(f"Accuracy: {accuracy[i]}")
        print(f"Sensitivity: {sensitivity[i]}")
        print(f"Specificity: {specificity[i]}")
        print(f"Precision: {precision[i]}")
        print(f"F1-score: {f1_score[i]}")
        print()

    # Micro measures
    tp_micro=tp.sum()
    fp_micro=fp.sum()
    tn_micro=tn.sum()
    fn_micro=fn.sum()

    accuracy_micro=(tp_micro+tn_micro)/(tp_micro+fp_micro+tn_micro+fn_micro)
    sensitivity_micro=tp_micro/(tp_micro+fn_micro) if (tp_micro + fn_micro) != 0 else np.nan
    specificity_micro=tn_micro/(tn_micro+fp_micro) if (tn_micro + fp_micro) != 0 else np.nan
    precision_micro=tp_micro/(tp_micro+fp_micro) if (tp_micro + fp_micro) != 0 else np.nan
    f1_score_micro=(2*tp_micro/(2*tp_micro+fn_micro+fp_micro)
        if not np.isnan(precision_micro)
        and not np.isnan(sensitivity_micro)
        and (precision_micro+sensitivity_micro) != 0
        else np.nan
    )

    print("Micro Performance Measures:")
    print(f"Accuracy: {accuracy_micro}")
    print(f"Sensitivity: {sensitivity_micro}")
    print(f"Specificity: {specificity_micro}")
    print(f"Precision: {precision_micro}")
    print(f"F1-score: {f1_score_micro}")
    print()

    # Macro measures
    accuracy_macro=np.nanmean(accuracy)
    sensitivity_macro=np.nanmean(sensitivity)
    specificity_macro=np.nanmean(specificity)
    precision_macro=np.nanmean(precision)
    f1_score_macro=np.nanmean(f1_score)

    print("Macro Performance Measures:")
    print(f"Accuracy: {accuracy_macro}")
    print(f"Sensitivity: {sensitivity_macro}")
    print(f"Specificity: {specificity_macro}")
    print(f"Precision: {precision_macro}")
    print(f"F1-score: {f1_score_macro}")
    print()

    # Weighted macro measures
    n=len(y_true)
    weights=np.array([(y_true == cls).sum() / n for cls in classes])

    accuracy_wmacro=np.nansum(weights*accuracy)
    sensitivity_wmacro=np.nansum(weights*sensitivity)
    specificity_wmacro=np.nansum(weights*specificity)
    precision_wmacro=np.nansum(weights*precision)
    f1_score_wmacro=np.nansum(weights*f1_score)

    print("Weighted Macro Performance Measures:")
    print(f"Accuracy: {accuracy_wmacro}")
    print(f"Sensitivity: {sensitivity_wmacro}")
    print(f"Specificity: {specificity_wmacro}")
    print(f"Precision: {precision_wmacro}")
    print(f"F1-score: {f1_score_wmacro}")
    print()

    return {
        "tp": tp,
        "fp": fp,
        "tn": tn,
        "fn": fn,
        "accuracy": accuracy,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "precision": precision,
        "f1_score": f1_score
    }

# ============================================
# READING DATA SET
# ============================================
movie_data=pd.read_csv(r"C:/Users/000110888/OneDrive - CSULB/Desktop/movie_data.csv")

# Encoding categorical variables into numeric
movie_data["gender"]=np.where(movie_data["gender"] == "M", 1, 0)
movie_data["member"]=np.where(movie_data["member"] == "yes", 1, 0)

movie_data["rating"]=movie_data["rating"].map({
    "very bad": 1,
    "bad": 2,
    "okay": 3,
    "good": 4,
    "very good": 5
})

# Min-max rescaling
movie_data["age"]=(movie_data["age"] - movie_data["age"].min()) / (
    movie_data["age"].max() - movie_data["age"].min()
)
movie_data["nmovies"]=(movie_data["nmovies"] - movie_data["nmovies"].min()) / (
    movie_data["nmovies"].max() - movie_data["nmovies"].min()
)

# ============================================
# CREATING TRAINING AND TESTING SETS
# ============================================
train, test=train_test_split(movie_data, test_size=0.2, stratify=movie_data["rating"],
random_state=187599)

# Display target value distribution
print(pd.DataFrame({
    "Count": train["rating"].value_counts().sort_index(),
    "Percentage": round(train["rating"].value_counts(normalize=True).sort_index() * 100, 2)
}))

print(pd.DataFrame({
    "Count": test["rating"].value_counts().sort_index(),
    "Percentage": round(test["rating"].value_counts(normalize=True).sort_index() * 100, 2)
}))

# Separating features and target
X_train=train.drop(columns=["rating"]).values
y_train=train["rating"].values
X_test=test.drop(columns=["rating"]).values
y_test=test["rating"].values

feature_names=train.drop(columns=["rating"]).columns

# ============================================
# RANDOM FOREST MULTINOMIAL CLASSIFIER
# ============================================
rf_mclass=RandomForestClassifier(n_estimators=150, max_features=4, max_leaf_nodes=30,
random_state=450024)
rf_mclass.fit(X_train, y_train)

rf_imp_df=pd.DataFrame({
    "Variable": feature_names,
    "MeanDecreaseGini": rf_mclass.feature_importances_
}).sort_values(by="MeanDecreaseGini", ascending=False)

print("Random Forest Multinomial Classifier - Feature Importance:")
print(rf_imp_df)

pred_class=rf_mclass.predict(X_test)

conf_mat=pd.DataFrame(confusion_matrix(y_test, pred_class, labels=[1, 2, 3, 4, 5]),
index=[1, 2, 3, 4, 5], columns=[1, 2, 3, 4, 5])
print("Random Forest Multinomial Classifier - Confusion Matrix:")
print(conf_mat)

print("Random Forest Performance Measures:")
performance_measures(pred_class, y_test)

# ============================================
# GRADIENT BOOSTING MULTINOMIAL CLASSIFIER
# ============================================
xgb_mclass=XGBClassifier(objective="multi:softmax", num_class=5, max_depth=6,
learning_rate=0.01, n_estimators=1000, random_state=558607, eval_metric="mlogloss")

# XGBoost requires labels 0 through 4
y_train_xgb=y_train-1
y_test_xgb=y_test-1

xgb_mclass.fit(X_train, y_train_xgb)

xgb_imp_df=pd.DataFrame({
    "Feature": feature_names,
    "Gain": xgb_mclass.feature_importances_
}).sort_values(by="Gain", ascending=False)

print("Gradient Boosting Multinomial Classifier - Feature Importance:")
print(xgb_imp_df)

pred_class=xgb_mclass.predict(X_test)+1

conf_mat=pd.DataFrame(confusion_matrix(y_test, pred_class, labels=[1, 2, 3, 4, 5]),
index=[1, 2, 3, 4, 5], columns=[1, 2, 3, 4, 5])
print("Gradient Boosting Multinomial Classifier - Confusion Matrix:")
print(conf_mat)

print("Gradient Boosting Performance Measures:")
performance_measures(pred_class, y_test)

# ============================================
# SVM WITH LINEAR KERNEL
# ============================================
svm_mclass_linear=SVC(kernel="linear", probability=True)
svm_mclass_linear.fit(X_train, y_train)

perm=permutation_importance(svm_mclass_linear, X_test, y_test, n_repeats=1, random_state=0)

importance = pd.DataFrame({
    "Variable": feature_names,
    "Importance": perm.importances_mean
}).sort_values(by="Importance", ascending=False)

print("SVM (Linear Kernel) Multinomial Classifier - Feature Importance:")
print(importance)

pred_class=svm_mclass_linear.predict(X_test)

conf_mat=pd.DataFrame(confusion_matrix(y_test, pred_class, labels=[1, 2, 3, 4, 5]),
index=[1, 2, 3, 4, 5], columns=[1, 2, 3, 4, 5])
print("SVM (Linear Kernel) Multinomial Classifier - Confusion Matrix:")
print(conf_mat)

print("SVM (Linear Kernel) Performance Measures:")
performance_measures(pred_class, y_test)

# ============================================
# SVM WITH POLYNOMIAL KERNEL
# ============================================
svm_mclass_poly=SVC(kernel="poly", probability=True)
svm_mclass_poly.fit(X_train, y_train)

perm=permutation_importance(svm_mclass_poly, X_test, y_test, n_repeats=1, random_state=0)

importance=pd.DataFrame({
    "Variable": feature_names,
    "Importance": perm.importances_mean
}).sort_values(by="Importance", ascending=False)

print("SVM (Polynomial Kernel) Multinomial Classifier - Feature Importance:")
print(importance)

pred_class=svm_mclass_poly.predict(X_test)

conf_mat=pd.DataFrame(
    confusion_matrix(y_test, pred_class, labels=[1, 2, 3, 4, 5]),
    index=[1, 2, 3, 4, 5],
    columns=[1, 2, 3, 4, 5]
)
print("SVM (Polynomial Kernel) Multinomial Classifier - Confusion Matrix:")
print(conf_mat)

print("SVM (Polynomial Kernel) Performance Measures:")
performance_measures(pred_class, y_test)

# ============================================
# SVM WITH RADIAL KERNEL
# ============================================
svm_mclass_radial=SVC(kernel="rbf", probability=True)
svm_mclass_radial.fit(X_train, y_train)

perm=permutation_importance(svm_mclass_radial, X_test, y_test, n_repeats=1, random_state=0)

importance=pd.DataFrame({
    "Variable": feature_names,
    "Importance": perm.importances_mean
}).sort_values(by="Importance", ascending=False)

print("SVM (Radial Kernel) Multinomial Classifier - Feature Importance:")
print(importance)

pred_class=svm_mclass_radial.predict(X_test)

conf_mat=pd.DataFrame(confusion_matrix(y_test, pred_class, labels=[1, 2, 3, 4, 5]),
index=[1, 2, 3, 4, 5], columns=[1, 2, 3, 4, 5])

print("SVM (Radial Kernel) Multinomial Classifier - Confusion Matrix:")
print(conf_mat)

print("SVM (Radial Kernel) Performance Measures:")
performance_measures(pred_class, y_test)

# ============================================
# SVM WITH SIGMOID KERNEL
# ============================================
svm_mclass_sigmoid=SVC(kernel="sigmoid", probability=True)
svm_mclass_sigmoid.fit(X_train, y_train)

perm=permutation_importance(svm_mclass_sigmoid, X_test, y_test, n_repeats=1, random_state=0)

importance=pd.DataFrame({
    "Variable": feature_names,
    "Importance": perm.importances_mean
}).sort_values(by="Importance", ascending=False)

print("SVM (Sigmoid Kernel) Multinomial Classifier - Feature Importance:")
print(importance)

pred_class=svm_mclass_sigmoid.predict(X_test)

conf_mat=pd.DataFrame(confusion_matrix(y_test, pred_class, labels=[1, 2, 3, 4, 5]),
index=[1, 2, 3, 4, 5], columns=[1, 2, 3, 4, 5])

print("SVM (Sigmoid Kernel) Multinomial Classifier - Confusion Matrix:")
print(conf_mat)

print("SVM (Sigmoid Kernel) Performance Measures:")
performance_measures(pred_class, y_test)

# ============================================
# KNN MULTINOMIAL CLASSIFIER
# ============================================
knn_mclass=KNeighborsClassifier()
knn_mclass.fit(X_train, y_train)

perm=permutation_importance(knn_mclass, X_test, y_test, n_repeats=1, random_state=845445)

importance=pd.DataFrame({
    "Variable": feature_names,
    "Importance": perm.importances_mean
}).sort_values(by="Importance", ascending=False)

print("KNN Multinomial Classifier - Feature Importance:")
print(importance)

pred_class=knn_mclass.predict(X_test)

conf_mat=pd.DataFrame(confusion_matrix(y_test, pred_class, labels=[1, 2, 3, 4, 5]),
index=[1, 2, 3, 4, 5], columns=[1, 2, 3, 4, 5])

print("KNN Multinomial Classifier - Confusion Matrix:")
print(conf_mat)

print("KNN Performance Measures:")
performance_measures(pred_class, y_test)

# ============================================
# NAIVE BAYES MULTINOMIAL CLASSIFIER
# ============================================
nb_mclass=GaussianNB()
nb_mclass.fit(X_train, y_train)

perm=permutation_importance(nb_mclass, X_test, y_test, n_repeats=1, random_state=0)

importance=pd.DataFrame({
    "Variable": feature_names,
    "Importance": perm.importances_mean
}).sort_values(by="Importance", ascending=False)

print("Naive Bayes Multinomial Classifier - Feature Importance:")
print(importance)

pred_class=nb_mclass.predict(X_test)

conf_mat=pd.DataFrame(confusion_matrix(y_test, pred_class, labels=[1, 2, 3, 4, 5]),
index=[1, 2, 3, 4, 5], columns=[1, 2, 3, 4, 5])

print("Naive Bayes Multinomial Classifier - Confusion Matrix:")
print(conf_mat)

print("Naive Bayes Performance Measures:")
performance_measures(pred_class, y_test)

# ============================================================
# FITTING ARTIFICIAL NEURAL NETWORK MULTINOMIAL CLASSIFIER
# ============================================================
from sklearn.neural_network import MLPClassifier
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

ann_mclass=MLPClassifier(hidden_layer_sizes=(3,), max_iter=2000, random_state=296707)
ann_mclass.fit(X_train, y_train)

# displaying probability-based permutation feature importance
baseline_prob=ann_mclass.predict_proba(X_test)

perm_imp=[]
for j in range(X_test.shape[1]):
    Xp=X_test.copy()
    np.random.shuffle(Xp[:, j])
    perm_prob=ann_mclass.predict_proba(Xp)
    perm_imp.append(np.mean(np.abs(perm_prob - baseline_prob)))

importance=pd.DataFrame({
    "Variable": feature_names,
    "Importance": perm_imp
}).sort_values(by="Importance", ascending=False)

print("ANN Multinomial Classifier - Feature Importance:")
print(importance)

# computing predicted classes for testing data
pred_class=ann_mclass.predict(X_test)

# displaying confusion matrix
conf_mat=pd.DataFrame(confusion_matrix(y_test, pred_class, labels=[1, 2, 3, 4, 5]),
index=[1, 2, 3, 4, 5], columns=[1, 2, 3, 4, 5])

print("ANN Multinomial Classifier - Confusion Matrix:")
print(conf_mat)

# displaying performance measures
print("ANN Performance Measures:")
ann_results=performance_measures(pred_class, y_test)